Content for and by IEEE Signal Processing Society. (Raul Valle & Contributors)

# Numerical Sequences and Series

[Real Number Systems](./Real_Number_Systems.ipynb) opened with a sequence $\{1, 1.4, 1.41, \dots\}$ that "tends to" $\sqrt{2}$ — and admitted we didn't yet know what that means. This workshop pays that debt: limits, Cauchy sequences, completeness, and the convergence tests that make infinite sums (and hence every transform in [DSP](../../Intro_DSP/README.md)) meaningful.

### Visual setup & helpers
Used throughout for convergence pictures. *(Safe to re-run anytime.)*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_sequence(a, title="", limit=None, eps=None, n_max=None):
    k = np.arange(1, len(a) + 1)
    plt.figure(figsize=(7.5, 2.6))
    plt.plot(k, a, ".", markersize=5)
    if limit is not None:
        plt.axhline(limit, color="k", linewidth=0.8, linestyle="--")
        if eps is not None:
            plt.fill_between(k, limit - eps, limit + eps, alpha=0.15,
                             label=f"±ε band (ε={eps})")
            plt.legend()
    if n_max: plt.xlim(0, n_max)
    plt.title(title); plt.xlabel("n"); plt.tight_layout(); plt.show()

---
### 🕐 Session 1 of 3 — *Sequences & Convergence* (~35 min)
**Goal:** define convergence rigorously; prove limits are unique; meet Cauchy sequences and completeness.
**Builds on:** [Basic Topology](./Basic_Topology.ipynb) Session 2 (compactness). &nbsp; **Feeds into:** Session 2 (limsup/liminf and special sequences).

---

## Convergent Sequences

💡 **Intuition.** "$a_n \to L$" is a *challenge game*. Skeptic names any tolerance $\varepsilon > 0$, no matter how tiny; you must name a time $N$ after which the sequence **never again** leaves the band $(L - \varepsilon, L + \varepsilon)$. If you can win for every $\varepsilon$, the sequence converges. Every convergence proof below is just a winning strategy: given $\varepsilon$, construct $N$.

**Definition.** In a metric space $(X, d)$, a sequence $\{p_n\}$ *converges* to $p \in X$ (written $p_n \to p$ or $\lim_{n\to\infty} p_n = p$) if

$$\forall \varepsilon > 0, \; \exists N \in \mathbb{N}: \; n \ge N \implies d(p_n, p) < \varepsilon$$

A sequence that does not converge *diverges*.

#### Visual: the ε-band game for $a_n = 1 + \frac{(-1)^n}{n}$

In [ ]:

# YOUR CODE HERE


**What just happened.** The points alternate above and below the dashed line at 1, and the shaded band is $\varepsilon = 0.1$ wide. Read off where the sequence enters it and never leaves: since $|a_n - 1| = 1/n$, we need $1/n < 0.1$, so **$N = 11$**. The skeptic played $\varepsilon = 0.1$; the answer was 11.

**The alternation is deliberate, and it is what the definition is built to handle.** The sequence approaches 1 from *both sides*, so it crosses the limit infinitely often and is never monotone. None of that matters. The definition asks only that the terms stay within the band eventually — not that they approach from one side, not that they get closer at every step. Students carrying a mental image of a sequence "climbing toward" its limit will mis-handle exactly this case.

**Play the game once more to see the structure.** For $\varepsilon = 0.01$, $N = 101$; for $\varepsilon = 0.001$, $N = 1001$. Here $N(\varepsilon) = \lceil 1/\varepsilon \rceil$ — a strategy that answers *every* challenge, which is precisely what convergence asserts. The band shrinks without limit and $N$ grows without limit, but $N$ is **finite for every $\varepsilon$**, and that finiteness is the entire content of the claim.

**Note what the picture cannot show, because this is where visual intuition fails.** No finite plot can distinguish this sequence from one that behaves identically for 60 terms and then jumps to 5. Convergence is a statement about all $n$ beyond every $N$, and no amount of plotting establishes it — the proof does. The figure is a way to *understand* the definition, not evidence for it.

**Worth naming the rate while it is on screen.** The error falls like $1/n$: to gain one decimal digit you need ten times as many terms. Fine as an illustration, hopeless as a numerical method. Compare the geometric series in Session 3, where the error falls like $x^n$ and each additional term buys a fixed factor. Both converge; only one is usable, and that distinction — convergence versus *rate* of convergence — is what separates a theorem from an algorithm.

### Proof: Limits are unique

💡 **Intuition.** If the sequence eventually huddles arbitrarily close to $p$ *and* to $p'$, then $p$ and $p'$ must be arbitrarily close to each other — and two *fixed* points at distance zero are the same point.

Suppose $p_n \to p$ and $p_n \to p'$. Let $\varepsilon > 0$. Choose $N_1$ with $d(p_n, p) < \varepsilon/2$ for $n \ge N_1$, and $N_2$ with $d(p_n, p') < \varepsilon/2$ for $n \ge N_2$. For any $n \ge \max(N_1, N_2)$, the triangle inequality gives

$$d(p, p') \le d(p, p_n) + d(p_n, p') < \tfrac{\varepsilon}{2} + \tfrac{\varepsilon}{2} = \varepsilon.$$

So $d(p, p') < \varepsilon$ for *every* $\varepsilon > 0$, forcing $d(p, p') = 0$, i.e. $p = p'$. $\blacksquare$

### Proof: Convergent sequences are bounded

Let $p_n \to p$ and take $\varepsilon = 1$: there is $N$ with $d(p_n, p) < 1$ for all $n \ge N$. Then every term lies within

$$r = \max\{1, \; d(p_1, p), \; \dots, \; d(p_{N-1}, p)\}$$

of $p$ — finitely many exceptional terms, each at a finite distance, plus a tail trapped in the unit ball. $\blacksquare$

## Subsequences

**Definition.** Given $\{p_n\}$ and indices $n_1 < n_2 < \cdots$, the sequence $\{p_{n_k}\}$ is a *subsequence*. $\{p_n\}$ converges to $p$ iff **every** subsequence converges to $p$.

**Theorem (Bolzano–Weierstrass).** Every bounded sequence in $\mathbb{R}^k$ has a convergent subsequence.

*Sketch:* a bounded sequence lives in a compact cell; by [Basic Topology](./Basic_Topology.ipynb) Session 2, an infinite subset of a compact set has a limit point in it, and we can extract a subsequence marching into that point.

## Cauchy Sequences & Completeness

💡 **Intuition.** Convergence has an annoying flaw as a *test*: you must already know the limit $L$ to check it. A **Cauchy** sequence replaces "terms get close to $L$" with "terms get close to *each other*" — checkable without knowing where you're headed. The deep fact about $\mathbb{R}$: these are the *same* sequences. That's completeness — the sequence-flavored restatement of the least-upper-bound property from [Real Number Systems](./Real_Number_Systems.ipynb). In $\mathbb{Q}$ it *fails*: $\{1, 1.4, 1.41, \dots\}$ is Cauchy but its target is missing.

**Definition.** $\{p_n\}$ is *Cauchy* if $\forall \varepsilon > 0, \exists N: \; m, n \ge N \implies d(p_n, p_m) < \varepsilon$.

### Proof: In $\mathbb{R}^k$, Cauchy $\iff$ convergent

($\Leftarrow$) If $p_n \to p$, pick $N$ with $d(p_n, p) < \varepsilon/2$ for $n \ge N$; then for $m, n \ge N$: $d(p_n, p_m) \le d(p_n, p) + d(p, p_m) < \varepsilon$. *(This direction holds in every metric space.)*

($\Rightarrow$) Let $\{p_n\}$ be Cauchy.
1. *Bounded:* take $\varepsilon = 1$; beyond some $N$ all terms are within $1$ of $p_N$, and only finitely many terms precede it.
2. *A convergent subsequence exists:* by Bolzano–Weierstrass, some $p_{n_k} \to p$.
3. *The whole sequence follows it:* given $\varepsilon$, choose $N$ so that $d(p_n, p_m) < \varepsilon/2$ for $m, n \ge N$, and pick a subsequence index $n_k \ge N$ with $d(p_{n_k}, p) < \varepsilon/2$. Then for every $n \ge N$:
$$d(p_n, p) \le d(p_n, p_{n_k}) + d(p_{n_k}, p) < \varepsilon. \;\blacksquare$$

---
### 🕐 Session 2 of 3 — *Monotone Sequences, limsup & Special Limits* (~35 min)
**Goal:** prove the monotone convergence theorem; define limsup/liminf; construct e as a limit.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (series).

---

## Monotone Sequences

💡 **Intuition.** A sequence that only climbs and has a ceiling has nowhere to go but *toward the ceiling it's straining against* — and the least-upper-bound property guarantees that ceiling is a real number. This is the workhorse theorem: it proves convergence with no idea what the limit is.

### Proof: Monotone Convergence Theorem

Let $a_n$ be increasing ($a_n \le a_{n+1}$) and bounded above. Put $L = \sup_n a_n$, which exists by the least-upper-bound property. Given $\varepsilon > 0$: $L - \varepsilon$ is not an upper bound, so some $a_N > L - \varepsilon$. By monotonicity, for all $n \ge N$:

$$L - \varepsilon < a_N \le a_n \le L \implies |a_n - L| < \varepsilon. \;\blacksquare$$

## Limit Superior & Inferior

💡 **Intuition.** A divergent sequence can still have organized behavior — oscillating between two rails. $\limsup$ is the height of the *upper* rail (the largest value hit infinitely often), $\liminf$ the lower. Convergence is exactly the collapse of the two rails onto one line.

**Definition.** For a bounded real sequence, $\limsup_{n} a_n = \lim_{n\to\infty} \left( \sup_{k \ge n} a_k \right)$ — the limit of a *decreasing* sequence of tail-suprema (which exists by monotone convergence!). Symmetrically for $\liminf$. Then $\{a_n\}$ converges iff $\limsup a_n = \liminf a_n$, and the common value is the limit.

#### Visual: the two rails of $a_n = (-1)^n (1 + \tfrac{1}{n})$

In [ ]:

# YOUR CODE HERE


**What just happened.** The dots never settle — odd terms sit near $-1$, even terms near $+1$, forever. The sequence plainly diverges. Yet the two curves squeezing it are perfectly well behaved: the red tail-supremum **decreases** to $1$ and the blue tail-infimum **increases** to $-1$, each monotonically, each converging.

**That monotonicity is the definition working, not an accident of this example.** $\sup_{k \ge n} a_k$ is a supremum over a shrinking set, so it can only decrease as $n$ grows; bounded below and decreasing, it converges by the monotone convergence theorem just proved. The same argument with inequalities reversed gives $\liminf$. So **limsup and liminf exist for every bounded sequence** — including ones with no limit at all. That is their entire purpose: a notion of limiting behaviour that never fails to be defined.

**Read the gap as a diagnosis.** Here $\limsup - \liminf = 2$, and that number *is* the amplitude of the persistent oscillation. Convergence is exactly the case where the gap closes to zero and the rails pinch onto one line. So $\limsup a_n = \liminf a_n$ is a convergence criterion requiring **no candidate limit** — the same virtue as the Cauchy criterion from Session 1, arrived at from a different direction.

**Note that neither rail is ever attained, and why that is the right definition.** The even terms are $1 + 1/n$, strictly *above* 1, and $\max_n a_n = 1.5$ at $n = 2$. So $\limsup = 1 \ne \sup = 1.5$. The limsup ignores any finite head of the sequence — it asks what values recur **infinitely often**, not what the largest value ever seen was. A single enormous early term shifts the sup and leaves the limsup untouched, which is precisely the robustness you want from a statement about tails.

**This is why the next session's tests are stated with limsup rather than lim.** The root test needs $\limsup \sqrt[n]{|a_n|}$ because for a series like $\sum c_n z^n$ with erratic coefficients, $\sqrt[n]{|c_n|}$ may oscillate and have no limit — but its limsup always exists, so the radius of convergence $R = 1/\limsup\sqrt[n]{|c_n|}$ is always defined. A test phrased with $\lim$ would simply be inapplicable to those series; phrased with $\limsup$ it never fails to give an answer.

## The Number $e$

### Proof: $\lim_{n\to\infty}\left(1 + \frac{1}{n}\right)^n$ exists

Let $a_n = (1 + \frac1n)^n$. By the binomial theorem,

$$a_n = \sum_{k=0}^{n} \binom{n}{k} \frac{1}{n^k} = \sum_{k=0}^{n} \frac{1}{k!} \prod_{j=0}^{k-1}\left(1 - \frac{j}{n}\right).$$

Each factor $(1 - j/n)$ *increases* with $n$ and the sum gains a term, so $a_n$ is increasing. Moreover every factor is $\le 1$, so

$$a_n \le \sum_{k=0}^n \frac{1}{k!} \le 1 + 1 + \frac12 + \frac1{2^2} + \cdots < 3,$$

using $k! \ge 2^{k-1}$. Increasing and bounded above ⇒ convergent, by monotone convergence. We *name* the limit $e$. $\blacksquare$

This is the same $e$ whose properties power the complex exponentials of [Foundations of Signal Processing §1.0](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb).

In [ ]:

# YOUR CODE HERE


**What just happened.** The column climbs monotonically — 2.000, 2.250, 2.488, 2.594, 2.705, 2.717 — exactly as the proof promised (increasing, bounded above by 3), and it is heading somewhere. But look at the last row against the target:

$$n = 10^6: \quad 2.71828047 \qquad e = 2.71828183$$

**A million terms and the seventh significant digit is already wrong.** The absolute error is $1.4 \times 10^{-6}$, which is no accident: expanding $\left(1+\tfrac1n\right)^n = \exp\!\big(n\log(1+\tfrac1n)\big) = \exp\!\big(1 - \tfrac{1}{2n} + O(n^{-2})\big)$ gives an error of about $e/(2n) = 1.36 \times 10^{-6}$ at $n = 10^6$. The measured gap is $1.36 \times 10^{-6}$. Theory and table agree to two figures.

**So the convergence is first-order, and that is a verdict on the formula as an algorithm.** Error $\propto 1/n$ means **ten times the work for one more decimal digit**. Reaching double precision, around $10^{-16}$, would need $n \approx 10^{16}$ terms — more arithmetic than any machine will do, and by then floating-point cancellation in $1 + 1/n$ has destroyed the answer anyway. The limit is real; the recipe for computing it is worthless.

**Which is the distinction worth carrying out of this session: convergent and *usable* are different claims.** The proof establishes that the limit exists — it says nothing whatever about how fast. Anyone actually computing $e$ uses $\sum_k 1/k!$, whose terms fall factorially: **17 terms** put you at machine precision, versus $10^{16}$ here. Same number, same theorem underwriting both, a factor of $10^{15}$ between them in cost.

**Note also what the proof did and did not do, because the logical order matters.** We showed the sequence is increasing and bounded above by 3, so monotone convergence guarantees a limit exists — *and then we named it $e$*. There was no prior object called $e$ that we approximated. This is the same move as defining $\sqrt2$ by a Cauchy sequence in [Real Number Systems](./Real_Number_Systems.ipynb): completeness lets you construct a number by cornering it, and the definition is legitimate precisely because the limit is guaranteed to be there.

**One honest caveat about the last row.** At $n = 10^6$, $1/n$ is near the edge of where double precision represents $1 + 1/n$ cleanly — about 10 significant digits survive the addition. Some of the discrepancy is arithmetic, not mathematics. Push to $n = 10^{16}$ and the computed value collapses to exactly 1.0, since $1 + 10^{-16}$ rounds to 1 — the sequence converges to $e$, but the *floating-point* sequence converges to 1.

---
### 🕐 Session 3 of 3 — *Series* (~40 min)
**Goal:** make sense of infinite sums; prove the comparison, geometric, root and ratio tests.
**Builds on:** Sessions 1–2. &nbsp; **Feeds into:** [DSP](../../Intro_DSP/README.md) — transforms are series; [Measure Theory](./README.md#4-measure-theory-draft--pending-review).

---

## Series

💡 **Intuition.** An infinite sum is *defined* as the limit of its partial sums — nothing more exotic. All the tests below are one idea in different clothes: compare your series against one whose fate you already know (usually the geometric series), and use monotone convergence on the partial sums.

**Definition.** Given $\{a_n\}$, the *series* $\sum a_n$ converges to $s$ if the partial sums $s_n = \sum_{k=1}^{n} a_k$ converge to $s$.

**Cauchy criterion for series:** $\sum a_n$ converges iff $\forall \varepsilon > 0, \exists N: \left| \sum_{k=n}^{m} a_k \right| < \varepsilon$ for all $m \ge n \ge N$. Taking $m = n$: **if $\sum a_n$ converges, then $a_n \to 0$** — necessary, famously not sufficient.

### Proof: The geometric series

For $|x| < 1$: $\;s_n = \sum_{k=0}^{n} x^k = \frac{1 - x^{n+1}}{1 - x}$ (multiply $s_n$ by $(1-x)$ and telescope). Since $x^{n+1} \to 0$ for $|x| < 1$,

$$\sum_{k=0}^{\infty} x^k = \frac{1}{1 - x}.$$

For $|x| \ge 1$ the terms don't tend to $0$, so the series diverges. $\blacksquare$

*This one formula computes the DTFT of the exponential decay — the most-used transform pair in [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb).*

### Proof: The harmonic series diverges (and $\sum 1/n^2$ doesn't)

💡 **Intuition.** Group the harmonic series into blocks that each add at least $\tfrac12$: $\tfrac13 + \tfrac14 > \tfrac12$, then $\tfrac15 + \cdots + \tfrac18 > \tfrac12$, and so on forever — infinitely many half-bricks stack to infinity. The terms go to zero, *but not fast enough*.

Partial sums over dyadic blocks: $\sum_{k=2^m+1}^{2^{m+1}} \frac{1}{k} \ge 2^m \cdot \frac{1}{2^{m+1}} = \frac{1}{2}$, so $s_{2^{m+1}} \ge 1 + \frac{m+1}{2} \to \infty$.

For $\sum \frac{1}{n^2}$, the same dyadic grouping gives blocks bounded by $2^m \cdot \frac{1}{(2^m)^2} = 2^{-m}$ — a convergent geometric series, so the increasing partial sums are bounded and converge. $\blacksquare$

In [ ]:

# YOUR CODE HERE


**What just happened.** Two series whose terms both go to zero, and two completely different fates. On the log axis the harmonic partial sums are a **straight line** — the signature of logarithmic growth, $s_n \approx \ln n + \gamma$, climbing without bound. The Basel sums flatten onto the dashed line at $\pi^2/6 \approx 1.6449$ and stop.

**Read the actual numbers, because they are the point.** After $10^5$ terms the harmonic sum has reached only about $\ln(10^5) + \gamma \approx 12.09$. Twelve. Ask the room how many terms are needed to reach 100: the answer is $e^{100} \approx 10^{43}$, more terms than there are atoms in the observable universe. **This series diverges, and no computation will ever show it.** Meanwhile the Basel sum is already within $10^{-5}$ of its limit — the tail beyond $N$ is about $1/N$.

**So the plot proves nothing, and that is the lesson worth taking.** Nothing on this screen distinguishes "grows like $\ln n$ forever" from "converges to something near 12". Numerically the two are indistinguishable at any $N$ you can compute. The dyadic block argument settles it in three lines; the experiment cannot settle it at all. If you ever wondered why analysis insists on proofs when a plot is available, this is the exhibit.

**And it kills the most common misconception in the subject.** $a_n \to 0$ is **necessary but not sufficient**. Both series pass that test; only one converges. The real question is never whether the terms vanish but *how fast*, and the dyadic grouping quantifies exactly that: harmonic blocks each contribute $\ge 1/2$ and there are infinitely many of them, while Basel blocks are bounded by $2^{-m}$ and sum geometrically.

**Notice the exponent is doing all the work — and where the boundary sits.** $\sum n^{-p}$ converges precisely for $p > 1$. Between $p = 1$ (divergent) and $p = 2$ (convergent) lies a sharp threshold at $p = 1$ itself, and the harmonic series sits exactly on the losing side of it. The same dyadic argument proves the general case: blocks are bounded by $2^m \cdot 2^{-pm} = 2^{(1-p)m}$, geometric with ratio $2^{1-p}$, which is under 1 exactly when $p > 1$. One argument, the whole family.

**A caveat on the $\pi^2/6$ line, in the spirit of honesty about what a demo shows.** The plot confirms convergence and is *consistent* with the limit being $\pi^2/6$ — it does not establish that value. Our dyadic argument bounds the sum by 2 and says nothing about which number below 2 it is. Euler's evaluation is a genuinely separate and much harder result; the Fourier-series proof of it becomes available after [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb), where Parseval applied to a sawtooth delivers it in a line.

### Comparison, Root & Ratio Tests

**Comparison.** If $|a_n| \le c_n$ eventually and $\sum c_n$ converges, then $\sum a_n$ converges (partial sums of $\sum |a_n|$ are increasing and bounded; absolute convergence implies convergence by the Cauchy criterion).

**Root test.** Let $\alpha = \limsup_n \sqrt[n]{|a_n|}$. If $\alpha < 1$: pick $\alpha < x < 1$; eventually $|a_n| < x^n$, and comparison with the geometric series finishes it. If $\alpha > 1$: $|a_n| > 1$ infinitely often, so $a_n \not\to 0$. ($\alpha = 1$: no information — both $\sum 1/n$ and $\sum 1/n^2$ have $\alpha = 1$.)

**Ratio test.** If $\limsup |a_{n+1}/a_n| < 1$, the terms are eventually dominated by a geometric sequence — converges. If the ratios are eventually $\ge 1$, terms can't die — diverges.

**Power series.** Applying the root test to $\sum c_n z^n$ gives convergence for $|z| < R = 1 / \limsup \sqrt[n]{|c_n|}$ — the *radius of convergence*, and precisely the "region of convergence" you will meet for the $z$-transform.

---
## Where next

- [Measure Theory](./README.md#4-measure-theory-draft--pending-review) — from summing sequences to integrating functions.
- [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — every transform there is a series whose convergence you can now interrogate.
- [Basic Topology](./Basic_Topology.ipynb) — compactness, which quietly powered Bolzano–Weierstrass above.